# 01c -- dim_fantasy_teams Seed

**Purpose**: Build `dim_fantasy_teams` from the league's Google Sheet owner registry.
The sheet is the single source of truth for team names, abbreviations, and
manager contacts. Cap state columns are initialized to 0 and overwritten by
ETL rollup from `fact_team` after rosters are loaded.

**Key**: `team_key` (matches Google Sheet "Team ID" column: A01-A14, B01-B14)

**Source**: Google Sheet — must be set to "Anyone with the link can view"

**Divisions → Conferences**:
- Riddell → A (A01–A14)
- Wilson  → B (B01–B14)

**Cap columns** (all 0 at seed time):

| Column | Formula |
|---|---|
| `original_cap` | `CFG.total_cap` — never changes |
| `cap_hits_current_yr` | ETL rollup from `fact_team` |
| `cap_hits_next_yr` | ETL rollup: year-2 contracts rolling forward |
| `reinvestment_cap` | In-season bonus cap charges |
| `active_roster_salary` | ETL rollup: active roster cap hits |
| `remaining_cap_current_yr` | `original_cap - (active_roster_salary + cap_hits_current_yr + reinvestment_cap)` |
| `remaining_cap_next_yr` | `original_cap - (projected_next_yr_salary + cap_hits_next_yr)` |

**Outputs**:
- `data/dim_fantasy_teams.parquet`

In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from datetime import date
from dataclasses import dataclass, field


@dataclass
class LeagueConfig:
    draft_year: int = 2026
    total_cap: int = 500_000_000
    num_teams: int = 28
    num_conferences: int = 2
    initial_contract_years: int = 3
    extension_contract_years: int = 3
    fa_minimum_salary: int = 2_000_000
    data_dir: str = "data"
    fuzzy_auto_threshold: int = 90
    fuzzy_review_threshold: int = 70
    # Google Sheet: Team/Owner registry -- export as CSV (sheet must be publicly viewable)
    team_sheet_id:  str = "1Fiz_KHH5bexSAHIfL0uVIqgHU6jTgnOmDs86kjR8TZc"
    team_sheet_gid: str = "178660131"

    @property
    def team_sheet_csv_url(self) -> str:
        return (
            f"https://docs.google.com/spreadsheets/d/{self.team_sheet_id}"
            f"/export?format=csv&gid={self.team_sheet_gid}"
        )


CFG = LeagueConfig()
DATA = Path(CFG.data_dir)
TODAY = date.today().isoformat()
DATA.mkdir(exist_ok=True)

## 1 -- Load from Google Sheet

Re-run this notebook any time team names or manager emails change in the sheet.

In [2]:
# -- Pull team registry from Google Sheet ------------------------------------
# Sheet must be set to "Anyone with the link can view".
# pd.read_csv follows redirects automatically via urllib.
print(f"Loading team sheet: {CFG.team_sheet_csv_url}")
try:
    teams_raw = pd.read_csv(CFG.team_sheet_csv_url)
except Exception as e:
    raise RuntimeError(
        f"Failed to load team sheet. Confirm sheet is publicly viewable.\n{e}"
    ) from e

print(f"Rows loaded : {len(teams_raw)}")
print(f"Columns     : {list(teams_raw.columns)}")
teams_raw

Loading team sheet: https://docs.google.com/spreadsheets/d/1Fiz_KHH5bexSAHIfL0uVIqgHU6jTgnOmDs86kjR8TZc/export?format=csv&gid=178660131
Rows loaded : 28
Columns     : ['Division', 'Team ID', 'Team Name', 'Team Abbreviation', 'Manager Email', 'Other Manager Email']


,Division,Team ID,Team Name,Team Abbreviation,Manager Email,Other Manager Email
0,Riddell,A01,AC/DeeCeedeez Nuts,tempfool,juventude85@yahoo.com,NaN
1,Riddell,A02,Cyndi Lauper (R),NT 5,davismelanie1999@gmail.com,NaN
2,Riddell,A03,Flava Flav (R),Bjuenger,brenden.juengert@gmail.com,NaN
3,Riddell,A04,JETHRO TULL (R),Thehardw,travnjazz1221@gmail.com,NaN
4,Riddell,A05,Kid Rock,K.R.,ray_jay@aol.com,NaN
5,Riddell,A06,Led Zeppelin (W),Bob 2,kannall73@gmail.com,NaN
6,Riddell,A07,Meat Loaf (R),Loaf,blord20@gmail.com,NaN
7,Riddell,A08,Metallica (W),Thebusdr,j.milakovich@yahoo.com,NaN
8,Riddell,A09,"""National Anthem""",DegenMik,mvreeland6@gmail.com,NaN
9,Riddell,A10,Pac & Big L Deadly Combo,P&L,benjaninja77@gmail.com,NaN


## 2 -- Build dim_fantasy_teams

In [3]:
# -- Column map: sheet -> dim_fantasy_teams -------------------------------------------
# Sheet columns: Division, Team ID, Team Name, Team Abbreviation,
#                Manager Email, Other Manager Email
_COL_MAP = {
    "Team ID":              "team_key",
    "Team Name":            "team_name",
    "Team Abbreviation":    "team_abbr",
    "Division":             "division",
    "Manager Email":        "manager_email",
    "Other Manager Email":  "manager_email_2",
}

# Division -> conference letter (Riddell = A, Wilson = B)
_DIV_CONF = {"Riddell": "A", "Wilson": "B"}


def build_dim_fantasy_teams(raw: pd.DataFrame, cfg: LeagueConfig) -> pd.DataFrame:
    # Validate expected columns
    missing = [c for c in _COL_MAP if c not in raw.columns]
    if missing:
        raise ValueError(f"Sheet missing expected columns: {missing}")

    df = raw.rename(columns=_COL_MAP).copy()

    # Derive conference from division
    df["conference"] = df["division"].map(_DIV_CONF)
    unmapped_divs = df[df["conference"].isna()]["division"].unique()
    if len(unmapped_divs):
        raise ValueError(f"Unknown divisions -- add to _DIV_CONF: {list(unmapped_divs)}")

    # Normalise nullable email column
    df["manager_email_2"] = df["manager_email_2"].where(df["manager_email_2"].notna(), other=pd.NA)

    # -- Cap state columns (all 0 at seed time; ETL rollup overwrites) -------
    orig_cap = cfg.total_cap
    df["original_cap"]             = orig_cap
    df["cap_hits_current_yr"]      = 0
    df["cap_hits_next_yr"]         = 0
    df["reinvestment_cap"]         = 0
    df["active_roster_salary"]     = 0
    # Derived: recomputed each ETL run
    # remaining_cap_current_yr = original_cap - (active_roster_salary + cap_hits_current_yr + reinvestment_cap)
    df["remaining_cap_current_yr"] = orig_cap - (
        df["active_roster_salary"] + df["cap_hits_current_yr"] + df["reinvestment_cap"]
    )
    # remaining_cap_next_yr = original_cap - (projected_next_yr_salary + cap_hits_next_yr)
    df["remaining_cap_next_yr"]    = orig_cap - (
        df["active_roster_salary"] + df["cap_hits_next_yr"]
    )

    # -- Final column order -----------------------------------------------
    cols = [
        "team_key", "team_name", "team_abbr",
        "conference", "division",
        "manager_email", "manager_email_2",
        "original_cap",
        "cap_hits_current_yr", "cap_hits_next_yr",
        "reinvestment_cap", "active_roster_salary",
        "remaining_cap_current_yr", "remaining_cap_next_yr",
    ]
    return df[cols].reset_index(drop=True)


dim_fantasy_teams = build_dim_fantasy_teams(teams_raw, CFG)

out_path = DATA / "dim_fantasy_teams.parquet"
dim_fantasy_teams.to_parquet(out_path, index=False)
print(f"dim_fantasy_teams: {len(dim_fantasy_teams)} teams -> {out_path}")
dim_fantasy_teams

dim_fantasy_teams: 28 teams -> data\dim_fantasy_teams.parquet


,team_key,team_name,team_abbr,conference,division,manager_email,manager_email_2,original_cap,cap_hits_current_yr,cap_hits_next_yr,reinvestment_cap,active_roster_salary,remaining_cap_current_yr,remaining_cap_next_yr
0,A01,AC/DeeCeedeez Nuts,tempfool,A,Riddell,juventude85@yahoo.com,NaN,500000000,0,0,0,0,500000000,500000000
1,A02,Cyndi Lauper (R),NT 5,A,Riddell,davismelanie1999@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
2,A03,Flava Flav (R),Bjuenger,A,Riddell,brenden.juengert@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
3,A04,JETHRO TULL (R),Thehardw,A,Riddell,travnjazz1221@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
4,A05,Kid Rock,K.R.,A,Riddell,ray_jay@aol.com,NaN,500000000,0,0,0,0,500000000,500000000
5,A06,Led Zeppelin (W),Bob 2,A,Riddell,kannall73@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
6,A07,Meat Loaf (R),Loaf,A,Riddell,blord20@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
7,A08,Metallica (W),Thebusdr,A,Riddell,j.milakovich@yahoo.com,NaN,500000000,0,0,0,0,500000000,500000000
8,A09,"""National Anthem""",DegenMik,A,Riddell,mvreeland6@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000
9,A10,Pac & Big L Deadly Combo,P&L,A,Riddell,benjaninja77@gmail.com,NaN,500000000,0,0,0,0,500000000,500000000


## 3 -- Validation

In [4]:
df = pd.read_parquet(DATA / "dim_fantasy_teams.parquet")
print(f"dim_fantasy_teams: {len(df)} rows, {len(df.columns)} columns")
print()

# Team count by conference / division
summary = df.groupby(["conference", "division"]).agg(
    teams=("team_key", "count"),
    missing_email=("manager_email", lambda x: x.isna().sum()),
).reset_index()
print("Teams by conference:")
print(summary.to_string(index=False))
print()

# Cap formula checks
ok_curr = (
    df["remaining_cap_current_yr"]
    == df["original_cap"]
       - (df["active_roster_salary"] + df["cap_hits_current_yr"] + df["reinvestment_cap"])
).all()
ok_next = (
    df["remaining_cap_next_yr"]
    == df["original_cap"] - (df["active_roster_salary"] + df["cap_hits_next_yr"])
).all()
print(f"remaining_cap_current_yr formula: {'OK' if ok_curr else 'MISMATCH'}")
print(f"remaining_cap_next_yr formula   : {'OK' if ok_next else 'MISMATCH'}")
print()

# Full roster display
display_cols = ["team_key", "team_name", "team_abbr", "conference", "division",
                "manager_email", "manager_email_2"]
print("Full team roster:")
print(df[display_cols].to_string(index=False))

dim_fantasy_teams: 28 rows, 14 columns

Teams by conference:
conference division  teams  missing_email
         A  Riddell     14              0
         B   Wilson     14              0

remaining_cap_current_yr formula: OK
remaining_cap_next_yr formula   : OK

Full team roster:
team_key                team_name team_abbr conference division              manager_email        manager_email_2
     A01       AC/DeeCeedeez Nuts  tempfool          A  Riddell      juventude85@yahoo.com                    NaN
     A02         Cyndi Lauper (R)      NT 5          A  Riddell davismelanie1999@gmail.com                    NaN
     A03           Flava Flav (R)  Bjuenger          A  Riddell brenden.juengert@gmail.com                    NaN
     A04          JETHRO TULL (R)  Thehardw          A  Riddell    travnjazz1221@gmail.com                    NaN
     A05                 Kid Rock      K.R.          A  Riddell            ray_jay@aol.com                    NaN
     A06         Led Zeppelin (W)  